# ACE-Step LoRA Training on Google Colab

Chuyển pipeline training LoRA của ACE-Step (repo fork woct0rdho) sang chạy trên Colab, tránh mấy vấn đề Windows-specific (Triton + MSVC, torchcodec/FFmpeg DLL, build gptqmodel...).

**Trước khi bắt đầu:** đảm bảo bạn đã tạo sẵn ở máy local: thư mục audio kèm các file `*_prompt.txt` và `*_lyrics.txt` tương ứng (từ bước `generate_prompts_lyrics.py`/`generate_prompts_lyrics_llamacpp.py`). Notebook này chỉ lo phần **preprocess + train**, không sinh lại prompt/lyrics.

**Trước khi chạy:** vào `Runtime > Change runtime type`, chọn GPU (T4 là đủ cho LoRA rank thấp; nếu có Colab Pro thì chọn A100/L4 để nhanh hơn).

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

Dùng Drive để lưu dataset (upload 1 lần) và checkpoint (persist qua các session, vì ổ đĩa VM của Colab bị xóa khi session kết thúc).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone repo

In [ ]:
%cd /content
!git clone https://github.com/woct0rdho/ACE-Step.git
%cd /content/ACE-Step

## 4. Cài dependencies

Colab đã có sẵn `torch`/`torchaudio`/`torchvision` bản CUDA phù hợp với GPU, nên **không** cài lại (tránh làm hỏng bản đã match driver). Cũng **không** cài `gptqmodel` và `av` từ `requirements.txt` gốc — hai gói đó chỉ cần cho bước sinh prompt/lyrics (không chạy ở đây), và `gptqmodel` bản cũ (2.2.0) sẽ xung đột với `peft` lúc training (peft yêu cầu gptqmodel >=5.6.12 nếu có cài, nên đơn giản nhất là không cài luôn).

In [ ]:
!pip install -q \
    "datasets==3.4.1" \
    "diffusers>=0.33.0" \
    "librosa==0.11.0" \
    "loguru==0.7.3" \
    "matplotlib==3.10.1" \
    numpy \
    "pypinyin==0.53.0" \
    "pytorch_lightning==2.5.1" \
    "soundfile==0.13.1" \
    tqdm \
    "transformers==4.52.3" \
    "py3langid==0.3.0" \
    "hangul-romanize==0.1.0" \
    "num2words==0.5.14" \
    "spacy==3.8.4" \
    "accelerate==1.6.0" \
    cutlet \
    "fugashi[unidic-lite]" \
    click \
    peft \
    tensorboard \
    tensorboardX \
    "h5py==3.13.0" \
    "hdf5plugin==5.1.0" \
    "huggingface-hub[hf_xet]==0.32.3" \
    "natsort==8.4.0" \
    "optimum==1.25.3" \
    "prodigyopt==1.1.2" \
    "requests==2.32.3" \
    "safetensors==0.5.3" \
    "wandb==0.20.1"

# Triton co san tren Colab (Linux), khong can cai them / khong can MSVC.
import torch
print('torch:', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 5. Upload dataset (audio + prompt + lyrics)

Ở máy local, nén thư mục `C:\data\audio` (chứa các file audio + `*_prompt.txt` + `*_lyrics.txt`) thành 1 file zip, ví dụ `ace_step_audio.zip`, rồi upload file đó lên Google Drive, thư mục gốc `My Drive` (hoặc đổi đường dẫn `ZIP_PATH` bên dưới cho khớp).

Chỉ cần làm 1 lần — những lần sau chỉ cần mount Drive và chạy lại cell giải nén.

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/ace_step_audio.zip'  # doi duong dan neu can

import os
os.makedirs('/content/data/audio', exist_ok=True)
!unzip -oq "$ZIP_PATH" -d /content/data/audio
!ls /content/data/audio | head -20
!echo "..." 
!ls /content/data/audio | wc -l

## 6. Tạo dataset chỉ chứa filenames

In [ ]:
!python convert2hf_dataset_new.py \
    --data_dir /content/data/audio \
    --output_name /content/data/audio_filenames

## 7. Preprocess (load audio, trich xuat feature, encode)

Bước này tự tải checkpoint gốc `ACE-Step/ACE-Step-v1-3.5B` cùng MERT/mHuBERT từ Hugging Face (lần đầu sẽ mất vài phút).

In [ ]:
!python preprocess_dataset_new.py \
    --input_name /content/data/audio_filenames \
    --output_dir /content/data/audio_prep

## 8. Cho checkpoint tự lưu vào Drive

`trainer_new.py` lưu LoRA vào thư mục cố định `./checkpoints`. Tạo symlink trỏ thư mục đó sang Drive để không mất khi Colab ngắt session.

In [ ]:
import os
drive_ckpt_dir = '/content/drive/MyDrive/ace_step_checkpoints'
os.makedirs(drive_ckpt_dir, exist_ok=True)

local_ckpt_link = '/content/ACE-Step/checkpoints'
if os.path.islink(local_ckpt_link) or os.path.exists(local_ckpt_link):
    if os.path.islink(local_ckpt_link):
        os.remove(local_ckpt_link)
else:
    os.symlink(drive_ckpt_dir, local_ckpt_link)

print('checkpoints ->', os.path.realpath(local_ckpt_link))

## 9. Train

`WANDB_MODE=offline` để tắt prompt tương tác (1/2/3) của wandb — cell chạy không tương tác được với input, nếu không set sẽ bị treo mãi chờ nhập. Chỉnh `--max_steps` tùy nhu cầu (mặc định 2000).

In [ ]:
import os
os.environ['WANDB_MODE'] = 'offline'

!python trainer_new.py \
    --dataset_path /content/data/audio_prep \
    --max_steps 2000

## 10. (Tuỳ chọn) Kiểm tra checkpoint đã lưu

In [ ]:
!ls -la /content/drive/MyDrive/ace_step_checkpoints

---
### Ghi chú
- Nếu Colab bị ngắt session giữa chừng: mount lại Drive, clone lại repo (hoặc `%cd` vào bản đã clone nếu VM chưa reset), cài lại dependencies, symlink lại checkpoints, rồi chạy lại `trainer_new.py` với `--last_lora_path` trỏ tới file `.safetensors` gần nhất trong `checkpoints/` để train tiếp thay vì train lại từ đầu.
- Nếu muốn theo dõi loss trực tiếp thay vì offline, đổi `WANDB_MODE` thành `online` và đăng nhập bằng API key (wandb.ai/authorize) trước khi train.
- Free tier Colab giới hạn giờ dùng GPU/tuần và có thể ngắt session bất kỳ lúc nào — nên lưu checkpoint thường xuyên (`--save_every_n_train_steps`, mặc định 100) và để symlink Drive như trên.